# Workable Code

In [7]:
%load_ext autoreload
%autoreload 2 

import pandas as pd
from src.utils import clean_customer_gdf_coordinates
from src.H3SpatialClusterer import H3SpatialClusterer  
import json
from src.get_data import DataFetcher, get_processed_data, get_geojson_data
import pickle
from src.utils import filter_cluster_result_dict
from src.plot_utils import plot_geojson_territory_heatmap

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Test Pipeline

In [8]:
## Fetch Data from DB
# fetcher = DataFetcher(logger=my_logger, input_dir="./input", sql_dir="_sql")
# results = fetcher.fetch_all()

In [9]:
OUTPUT_JSON_MAP = "./output/h3_clusters_map.geojson"
OUTPUT_JSON_MAP_FILTER = "./output/h3_clusters_map_filter.geojson"
OUTPUT_RESULTS_PICKLE = "./output/cluster_results.pickle"
OUTPUT_MAP_FILTERED = './output/map/test_nigeria_clusters_filter.html' 

# CUSTOMER_DENSITY_THRESHOLD_HIGH = 200
# CUSTOMER_DENSITY_THRESHOLD_LOW = 10
# MAX_H3_RESOLUTION_URBAN = 12
# MERGE_SEARCH_RADIUS = 1
# POP_DENSITY_HIGH = 5000
# POP_DENSITY_MEDIUM = 1000

# sp_dim_df.query("stock_point_name in ['OmniHub Oyigbo Rivers - LAMDA GLOBAL','OmniHub Apapa Lagos - CAUSEWAY','OmniHub Egbeda Oyo - Vizazi']")#['stock_point_name'].values[0]
# lgas_gdf.query('state_name == "Oyo"').area_km2.sum()
pilot_sps_lists = ['1647024','1647113','1647122']
stock_point_id = str(pilot_sps_lists[1])

In [10]:
#### Get Data
lgas_gdf,  sp_dim_df,  stock_point_lga_map, customers_gdf = get_processed_data()

⚠️ Warning: Dropped 806 customer records with coordinates outside Nigeria.
✅ Customer data cleaning complete. 28209 of 29015 records retained.


In [11]:
# sp_dim_df.columns
customers_gdf.columns

Index(['customer_id', 'stock_point_id', 'longitude', 'latitude', 'geometry'], dtype='object')

In [12]:
# 1. Initialize with real dataframes
clusterer = H3SpatialClusterer(
    lga_gdf=lgas_gdf,
    sp_dim_df=sp_dim_df, 
    stock_point_lga_map=stock_point_lga_map,
    customers_gdf=customers_gdf
)

✅ H3SpatialClusterer initialized.
📊 Data summary: 114 LGAs, 75 stock points, 28209 customers


In [13]:
# # 3. Run complete pipeline
'''
This will return a dictionary with the following keys: ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']
1. territories: A dictionary of stock point territories 
        # Dict[stock_point_id, {
            'polygon': Union[Polygon, MultiPolygon],
            'lga_ids': List[str],
            'is_contiguous': bool,
            'sub_territories': List[Polygon],
            'total_area_km2': float,
            'territory_version': str
        }]

2. grid_results: A dictionary of grid results for each territory
        Dict[stock_point_id, {
                'h3_resolution': int,
                'h3_cells': Set[str],
                'clipped_cells': Set[str], 
                'cell_geometries': Dict[str, Polygon],
                'territory_coverage': float
        }]
        
3. assignments: A dictionary of customer assignments to stock points
        Dict[stock_point_id, assignments_gdf with columns:
                ['customer_id', 
                'cluster_id', 
                'h3_cell_id', 
                'assignment_confidence', 
                'assignment_tier',
                'geometry']]
        
4. optimized_clusters: A dictionary of optimized clusters for each territory

5. statistics: A dictionary of statistics for each territory

6. territory_version: The version of the territory used in the clustering
'''

# # RESULTS: ETA 2MINS
# results = clusterer.process_all_stock_points(territory_version="v1.2")

# # Save Results as pickle file
# with open(OUTPUT_RESULTS_PICKLE, 'wb') as f:
#     pickle.dump(results, f)


"\nThis will return a dictionary with the following keys: ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version']\n1. territories: A dictionary of stock point territories \n        # Dict[stock_point_id, {\n            'polygon': Union[Polygon, MultiPolygon],\n            'lga_ids': List[str],\n            'is_contiguous': bool,\n            'sub_territories': List[Polygon],\n            'total_area_km2': float,\n            'territory_version': str\n        }]\n\n2. grid_results: A dictionary of grid results for each territory\n        Dict[stock_point_id, {\n                'h3_resolution': int,\n                'h3_cells': Set[str],\n                'clipped_cells': Set[str], \n                'cell_geometries': Dict[str, Polygon],\n                'territory_coverage': float\n        }]\n        \n3. assignments: A dictionary of customer assignments to stock points\n        Dict[stock_point_id, assignments_gdf with columns:\n         

In [14]:
# Read results from the pickle file
with open(OUTPUT_RESULTS_PICKLE, 'rb') as f:
    results = pickle.load(f)

filtered_result = filter_cluster_result_dict(results, pilot_sps_lists)

In [15]:
# 4. Export for deployment: ETA: 1 Min
csv_files = clusterer.export_results(results, output_format="csv")

# Stock Point Assignment Summary
df_output_assignment = csv_files['assignments']
df_output_cluster = csv_files['clusters']

sp_assignment_summary = (df_output_assignment
                            .groupby(['stock_point_id','cluster_id'])['customer_id'].count()
                            .reset_index(name='n_customers')
                            .rename({'cluster_id':'cell'}, axis=1)
                        )

# Stock Point Coverage - Assignment Summary
sp_coverage_assignment_summary = (csv_files['territory_cells']
                                .merge(sp_assignment_summary, how='left', on=['stock_point_id','cell'])
                                .fillna({'n_customers':0})
                                ) 







In [16]:


# csv_files['clusters'].to_csv('./output/clusters_output.csv', index=False)
# csv_files['clusters'].to_feather('./output/clusters_output.feather') 

# csv_files['assignments'].to_csv('./output/customer_assignments.csv', index=False)
# csv_files['assignments'].to_feather('./output/customer_assignments.feather')

# sp_coverage_assignment_summary.to_feather('./output/sp_coverage_assignment_summary.feather' )
# sp_assignment_summary.to_feather('./output/sp_assignment_summary.feather' )

In [17]:
sp_dim_df.head(1)

# df_output_cluster.sample(10)

,stock_point_id,stock_point_name,latitude,longitude
0,1646941,OmniHub Ido Oyo - CARESGATE AFRICA LTD,7.353881,3.836766


In [18]:
customers_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 28209 entries, 0 to 29014
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype   
---  ------          --------------  -----   
 0   customer_id     28209 non-null  int64   
 1   stock_point_id  28209 non-null  int64   
 2   longitude       28209 non-null  float64 
 3   latitude        28209 non-null  float64 
 4   geometry        28209 non-null  geometry
dtypes: float64(2), geometry(1), int64(2)
memory usage: 1.3 MB


In [19]:
# sp_dim_df.columns

# customers_gdf.columns

# df_output_assignment.columns

    # sp_dim_df: Columns: ['stock_point_id', 'stock_point_name', 'latitude', 'longitude']
    # customers_gdf: Columns:['customer_id',  'longitude', 'latitude', 'geometry'] # NB geometry is shapely POINT (lon, lat)
    # df_output_assignment: columns:['stock_point_id', 'customer_id', 'cluster_id', 'h3_cell_id', 'assignment_confidence', 'assignment_tier']

# sp_assignment_summary.columns
# sp_assignment_summary: ['stock_point_id', 'cell', 'n_customers']

# sp_coverage_assignment_summary.columns 
# sp_coverage_assignment_summary: ['cell', 'stock_point_id', 'h3_resolution', 'n_customers']

In [20]:
# 5. Export for GeoJSON: ETA: 1 Min

# sql_statements and geojson_map
geojson_map = clusterer.export_results(results, output_format="geojson")
geojson_map_filtered = clusterer.export_results(filtered_result, output_format="geojson") 

# Save full geojson_map data
try:
    with open(OUTPUT_JSON_MAP, 'w') as f:
        f.write(geojson_map)
    print(f"GeoJSON map successfully saved to {OUTPUT_JSON_MAP}")
except IOError as e:
    print(f"Error saving GeoJSON map to file: {e}")
    

# Save filtered geojson_map data    
try:
    with open(OUTPUT_JSON_MAP_FILTER, 'w') as f:
        f.write(geojson_map_filtered)
    print(f"GeoJSON map successfully saved to {OUTPUT_JSON_MAP_FILTER}")
except IOError as e:
    print(f"Error saving GeoJSON map to file: {e}")
    


GeoJSON map successfully saved to ./output/h3_clusters_map.geojson
GeoJSON map successfully saved to ./output/h3_clusters_map_filter.geojson


In [21]:
## PLOT 
 
_ = plot_geojson_territory_heatmap(geojson_path = OUTPUT_JSON_MAP_FILTER, output_path = OUTPUT_MAP_FILTERED)

Successfully loaded GeoJSON data from ./output/h3_clusters_map_filter.geojson


In [24]:
import folium
from folium.plugins import MeasureControl
# Add markers for each stock point
m = _

for idx, row in sp_dim_df.query(f'stock_point_id in {[int(sp) for sp in  pilot_sps_lists]}').iterrows():
    folium.Marker(
        location=[row['latitude'], row['longitude']],
        popup=f"ID: {row['stock_point_id']}<br>Name: {row['stock_point_name']}",
        tooltip=f"{row['stock_point_name']} (ID: {row['stock_point_id']})"
    ).add_to(_)

# Display the map
m.add_child(MeasureControl())
m

In [27]:
m.save(f'./output/map/selected_stockpoints_assignment.html')

In [ ]:
# 5. Get processing summary
summary = clusterer.get_processing_summary()
summary

In [ ]:
# # 6. Deploy to database
# for sql in sql_statements:
#     database.execute(sql)

# # 7. Save visualization files
# with open('nigeria_clusters.geojson', 'w') as f:
#     f.write(geojson_map) 

In [ ]:
summary

## Exploratory Analysis

In [ ]:
# results.get('grid_results', {}).get(stock_point_id, {}).keys()
# results.get('grid_results', {}).get(stock_point_id, {}).get('cell_geometries', {})#keys()
csv_files['clusters'].query('stock_point_id == @stock_point_id').head(1)
sp_coverage_assignment_summary.query('stock_point_id == @stock_point_id').head(1)
sp_assignment_summary.query('stock_point_id == @stock_point_id').head(1)

### Diagonistics

In [ ]:
# stock_point_id = str(pilot_sps_lists[0])
# results['optimized_clusters'][stock_point_id].keys()
# ['territories', 'grid_results', 'assignments', 'optimized_clusters', 'statistics', 'territory_version'] 

territories = results['territories'][stock_point_id]
# ['polygon', 'lga_ids', 'is_contiguous', 'sub_territories', 
# 'total_area_km2', 'territory_version', 'lga_count', 'validation_status']

grid_data = results['grid_results'][stock_point_id]
# ['h3_resolution', 'h3_cells', 'clipped_cells', 'cell_geometries', 'territory_coverage']

assignments = results['assignments'][stock_point_id]
# ['customer_id', 'cluster_id', 'h3_cell_id', 'assignment_confidence',  'assignment_tier', 'geometry']

optimized_clusters = results['optimized_clusters'][stock_point_id]
# statistics = results['statistics'][stock_point_id]
# territory_version = results['territory_version']
# results['territories'][stock_point_id]
# results['assignments'][stock_point_id].sample(2)
# results['optimized_clusters'][stock_point_id]#.sample(2)

In [ ]:
# assignment cell summary: Dataframe: cluster_id, n_cusomter
# TO-DO ADD SUMMARY BY 
try:
    assignment_cell_summary =  assignments.groupby('cluster_id')['customer_id'].count().reset_index(name='n_customers')
except:
    assignment_cell_summary = pd.DataFrame(columns = ['cluster_id','n_customers'])

In [ ]:
import geopandas as gpd
import pandas as pd

def reassign_customers_with_optimized_clusters(
    assignments: Dict[str, gpd.GeoDataFrame],
    optimized_clusters: Dict[str, pd.DataFrame]
) -> Dict[str, gpd.GeoDataFrame]:
    """
    Reassigns customers to optimized cluster IDs based on the output of optimize_clusters.
    
    Args:
        assignments: Dict mapping stock_point_id to GeoDataFrame with customer data
                    (columns include 'cluster_id', 'geometry').
        optimized_clusters: Dict mapping stock_point_id to DataFrame with optimized clusters
                           (columns: 'cluster_id', 'h3_resolution', 'h3_cells', 'customer_count', 'parent_cluster_id').
    
    Returns:
        Dict[stock_point_id, GeoDataFrame with updated 'cluster_id' for each customer].
    """
    print("🔄 Reassigning customers to optimized cluster IDs...")
    
    reassigned_assignments = {}
    
    for stock_point_id, assignments_gdf in assignments.items():
        print(f"Processing stock point {stock_point_id}...")
        
        if assignments_gdf.empty or stock_point_id not in optimized_clusters:
            reassigned_assignments[stock_point_id] = assignments_gdf.copy()
            continue
        
        optimized_df = optimized_clusters[stock_point_id]
        if optimized_df.empty:
            reassigned_assignments[stock_point_id] = assignments_gdf.copy()
            continue
        
        # Create a mapping from original H3 cell to optimized cluster_id
        cell_to_cluster_id = {}
        for _, cluster in optimized_df.iterrows():
            cluster_id = cluster['cluster_id']
            for cell in cluster['h3_cells']:
                cell_to_cluster_id[cell] = cluster_id
        
        # Copy the input GeoDataFrame to preserve all columns
        reassigned_gdf = assignments_gdf.copy()
        
        # Reassign cluster_id for each customer
        reassigned_gdf['optimized_cluster_id'] = None
        for idx, customer in reassigned_gdf.iterrows():
            original_cell = customer['cluster_id']
            if original_cell in cell_to_cluster_id:
                reassigned_gdf.at[idx, 'optimized_cluster_id'] = cell_to_cluster_id[original_cell]
            else:
                # If the original cell isn't in optimized clusters, try reassigning based on location
                lat, lng = customer.geometry.y, customer.geometry.x
                customer_resolution = optimized_df['h3_resolution'].max()  # Use highest resolution
                customer_cell = h3.latlng_to_cell(lat, lng, customer_resolution)
                # Find the optimized cluster containing this cell
                for _, cluster in optimized_df.iterrows():
                    if customer_cell in cluster['h3_cells'] or h3.cell_to_parent(customer_cell, cluster['h3_resolution']) in cluster['h3_cells']:
                        reassigned_gdf.at[idx, 'optimized_cluster_id'] = cluster['cluster_id']
                        break
                if reassigned_gdf.at[idx, 'optimized_cluster_id'] is None:
                    print(f"⚠️ Customer at index {idx} could not be reassigned, keeping original cluster_id {original_cell}")
                    reassigned_gdf.at[idx, 'optimized_cluster_id'] = original_cell
        
        reassigned_assignments[stock_point_id] = reassigned_gdf
        print(f"✅ Reassigned {len(reassigned_gdf)} customers for stock point {stock_point_id}")
    
    print("🏁 Customer reassignment complete")
    return reassigned_assignments

In [ ]:
# Get LGA IDs for this stock point
stock_point_id = 1647108
lga_ids = clusterer.stock_point_lga_map[
    clusterer.stock_point_lga_map['stock_point_id'] == stock_point_id
]['lga_id'].tolist()

In [ ]:
# lga_ids
if not lga_ids:
    print(f"⚠️ Warning: No LGAs found for stock_point_id {stock_point_id}")
    # continue

# Get LGA geometries
territory_lgas = clusterer.lgas[clusterer.lgas['lga_id'].isin(lga_ids)].copy()
territory_lgas

In [ ]:
lga_gdf.shape # (115, 18)

lga_gdf_2 = lga_gdf.dropna(subset=['geometry'])

lga_gdf_2[lga_gdf_2['lga_id'].isin(lga_ids)].copy()